In [1]:
!pip install wandb

In [2]:
import wandb

wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rachin15-5466 (rachin15-5466-daffodil-international-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [24]:
from wandb.integration.keras import WandbMetricsLogger
import tensorflow as tf
from tensorflow import keras

In [25]:
sweep_config = {

    'method' : 'grid',
    'metric' : {
        'name': 'val_accuracy',
        'goal': 'maximize'
                },
    'parameters' : {
        'batch_size': {'values': [8,16]},
        'learning_rate': {'values': [0.001,0.0001]},
        'hidden_nodes': {'values': [128,64]},
        'img_size': {'values': [16, 224]},
        'epochs': {'values': [5,10]}
    }
}

sweep_id = wandb.sweep(sweep_config, project="weight-and-bias")

Create sweep with ID: jfrk31xk
Sweep URL: https://wandb.ai/rachin15-5466-daffodil-international-university/weight-and-bias/sweeps/jfrk31xk


In [26]:
DATA_PATH = "/content/drive/MyDrive/ALL DATASET/rooms_dataset"

In [27]:
def train():
  with wandb.init() as run:
    config = wandb.config

      # Constants
    IMG_HEIGHT = config.img_size
    IMG_WIDTH = config.img_size
    IMG_CHANNELS = 3
    CLASS_NAMES = ["bed_room", "living_room", "dining_room"]


    train_dataset = tf.keras.utils.image_dataset_from_directory(
        DATA_PATH,
        validation_split=0.2,
        subset="training",
        seed=42,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=config.batch_size
    )


    eval_dataset = tf.keras.utils.image_dataset_from_directory(
        DATA_PATH,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=config.batch_size
    )

    train_dataset = train_dataset.prefetch(
        tf.data.AUTOTUNE
    )

    eval_dataset = eval_dataset.prefetch(
        tf.data.AUTOTUNE
    )


    model = keras.Sequential([

        keras.layers.Rescaling(1./255, input_shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS )),

        keras.layers.Flatten(),

        keras.layers.Dense( config.hidden_nodes, activation="relu" ),

        keras.layers.Dense( len(CLASS_NAMES), activation="softmax" )
    ])

    model.compile(loss = "sparse_categorical_crossentropy", optimizer = "adam", metrics = ["accuracy"])

    model.fit(train_dataset, validation_data = eval_dataset, epochs = config.epochs, callbacks = [WandbMetricsLogger(log_freq=5)])


In [28]:
wandb.agent(sweep_id, function=train)

wandb: Agent Starting Run: kfrr4ipp with config:
wandb: 	batch_size: 8
wandb: 	epochs: 5
wandb: 	hidden_nodes: 128
wandb: 	img_size: 16
wandb: 	learning_rate: 0.001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Found 118 files belonging to 3 classes.
Using 95 files for training.
Found 118 files belonging to 3 classes.
Using 23 files for validation.
Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


12/12 ━━━━━━━━━━━━━━━━━━━━ 21s 2s/step - accuracy: 0.3684 - loss: 1.4211 - val_accuracy: 0.3478 - val_loss: 1.7055
Epoch 2/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 61ms/step - accuracy: 0.5158 - loss: 1.1416 - val_accuracy: 0.5217 - val_loss: 1.0306
Epoch 3/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 65ms/step - accuracy: 0.5158 - loss: 1.0114 - val_accuracy: 0.4783 - val_loss: 1.0352
Epoch 4/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 2s 100ms/step - accuracy: 0.5263 - loss: 1.0015 - val_accuracy: 0.3043 - val_loss: 1.1131
Epoch 5/5
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 103ms/step - accuracy: 0.5789 - loss: 0.9018 - val_accuracy: 0.4783 - val_loss: 0.9344


batch/accuracy,▁▃▃▃▆▆▃▇▆▆█▆▃▆█
batch/batch_step,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,▂▄▄█▃▂▂▁▂▁▁▂▂▁▁
epoch/accuracy,▁▆▆▆█
epoch/epoch,▁▃▅▆█
epoch/learning_rate,▁▁▁▁▁
epoch/loss,█▄▂▂▁
epoch/val_accuracy,▂█▇▁▇
epoch/val_loss,█▂▂▃▁
batch/accuracy,0.57955


...

batch/accuracy,▁▃▄▄▄▄▄▅▂▆▄▅▇█▅▇▄▇▆█
batch/batch_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,▁█▃▃▄▃▃▂▂▂▂▂▂▁▁▁▂▁▁▁
epoch/accuracy,▁▃▂▄▅▄█▆▇█
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▃▂▂▂▁▁▁▁
epoch/val_accuracy,▄▁▆▂▁█▂▂▃▄
epoch/val_loss,▃█▅▂▃▂▂▂▁▁
batch/accuracy,0.63158


wandb: Agent Starting Run: 48ebpcjb with config:
wandb: 	batch_size: 16
wandb: 	epochs: 10
wandb: 	hidden_nodes: 64
wandb: 	img_size: 224
wandb: 	learning_rate: 0.0001
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


Found 118 files belonging to 3 classes.
Using 95 files for training.
Found 118 files belonging to 3 classes.
Using 23 files for validation.
Epoch 1/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 429ms/step - accuracy: 0.3684 - loss: 62.3534 - val_accuracy: 0.4348 - val_loss: 62.2618
Epoch 2/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 424ms/step - accuracy: 0.2947 - loss: 49.0691 - val_accuracy: 0.2174 - val_loss: 12.4032
Epoch 3/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 363ms/step - accuracy: 0.4000 - loss: 11.4473 - val_accuracy: 0.4348 - val_loss: 8.5274
Epoch 4/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 258ms/step - accuracy: 0.5158 - loss: 4.3937 - val_accuracy: 0.3913 - val_loss: 3.8723
Epoch 5/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 253ms/step - accuracy: 0.5789 - loss: 2.6087 - val_accuracy: 0.7391 - val_loss: 1.0841
Epoch 6/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 2s 254ms/step - accuracy: 0.7053 - loss: 1.1273 - val_accuracy: 0.5217 - val_loss: 1.3510
Epoch 7/10
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 276ms/step - accuracy: 0.7684 - loss: 0.6254 - val_accuracy:

batch/accuracy,▁▂▁▁▂▂▂▄▅▅█▆▆▇██▅▇▅█
batch/batch_step,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
batch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
batch/loss,▁█▅▆▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/accuracy,▂▁▂▄▅▆▇█▇█
epoch/epoch,▁▂▃▃▄▅▆▆▇█
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▆▂▁▁▁▁▁▁▁
epoch/val_accuracy,▄▁▄▃█▅█▅▆█
epoch/val_loss,█▂▂▁▁▁▁▁▁▁
batch/accuracy,0.81053


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.
